# Chapter 5: First-Order Logic

```{admonition} Learning Objectives
:class: tip
- Understand first-order logic syntax: predicates, functions, quantifiers
- Master quantifier semantics and scope
- Implement unification algorithm
- Apply resolution in first-order logic
- Build forward and backward chaining for FOL
- Design knowledge bases with rules and facts
- Create Prolog-style logic programs
```

```{epigraph}
First-order logic is to propositional logic as calculus is to arithmetic.

-- Stuart Russell & Peter Norvig
```

## 5.1 Introduction

First-Order Logic (FOL), also called **Predicate Logic** or **First-Order Predicate Calculus**, extends propositional logic with powerful new constructs.

### Limitations of Propositional Logic

**Problem**: Cannot express general statements about collections of objects.

**Example**: "All humans are mortal"

In propositional logic:
- Need separate propositions for each human
- HumanSocrates → MortalSocrates
- HumanPlato → MortalPlato
- HumanAristotle → MortalAristotle
- ...infinitely many statements!

**Solution**: First-Order Logic introduces:
1. **Objects**: Things in the world (Socrates, Plato, 5, red)
2. **Predicates**: Properties and relations (Human, Mortal, GreaterThan)
3. **Functions**: Mappings between objects (fatherOf, sqrt)
4. **Variables**: Placeholders for objects (x, y, z)
5. **Quantifiers**: "For all" (∀) and "There exists" (∃)

Now we can write:
$$\forall x: Human(x) \rightarrow Mortal(x)$$

This single sentence expresses infinitely many facts!

### Why FOL Matters

**Applications**:
- Expert systems with complex rules
- Database query languages (SQL is based on FOL)
- Semantic web and knowledge graphs
- Natural language understanding
- Automated theorem proving
- Logic programming (Prolog)

**Power**:
- Express relationships between objects
- Quantify over infinite domains
- Natural representation of knowledge
- Foundation for most knowledge representation systems

In [ ]:
from typing import List, Set, Dict, Tuple, Optional, Any, Union
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
from collections import defaultdict
from copy import deepcopy
import re

print('✓ Libraries imported successfully')

## 5.2 First-Order Logic Syntax

### Terms

A **term** represents an object in the domain.

**Types of Terms**:

1. **Constants**: Specific objects
   - Examples: Socrates, 5, Stanford, Red
   - Convention: Start with capital letter or number

2. **Variables**: Placeholders for objects
   - Examples: x, y, person, number
   - Convention: Start with lowercase letter

3. **Functions**: Mappings from objects to objects
   - Examples: fatherOf(John), sqrt(16), successor(5)
   - Can be nested: fatherOf(fatherOf(John))

### Atomic Sentences

**Predicate**: Relation among objects (evaluates to true/false)

**Atomic Sentence**: Predicate applied to terms
- Examples:
  - Human(Socrates)
  - GreaterThan(5, 3)
  - Parent(John, Mary)
  - Loves(John, Mary)

**Arity**: Number of arguments a predicate takes
- Human/1 (unary)
- GreaterThan/2 (binary)
- Between/3 (ternary)

### Complex Sentences

Built using logical connectives (same as propositional logic):
- ¬ (NOT)
- ∧ (AND)
- ∨ (OR)
- → (IMPLIES)
- ↔ (IFF)

Plus **quantifiers**:

1. **Universal Quantifier** (∀): "for all"
   - ∀x: P(x) means "P(x) is true for all x"
   - Example: ∀x: Human(x) → Mortal(x)

2. **Existential Quantifier** (∃): "there exists"
   - ∃x: P(x) means "P(x) is true for at least one x"
   - Example: ∃x: King(x) ∧ Evil(x)

### Quantifier Scope

**Scope**: The part of the formula the quantifier applies to

Examples:
- ∀x: (P(x) → Q(x)) - scope is entire implication
- (∀x: P(x)) → Q(x) - scope is just P(x)

**Free vs. Bound Variables**:
- **Bound**: Within scope of quantifier
- **Free**: Not bound by any quantifier

Example: ∀x: P(x, y)
- x is bound
- y is free

In [ ]:
# FOL Expression Classes

class Term(ABC):
    """Base class for FOL terms."""
    
    @abstractmethod
    def __str__(self) -> str:
        pass
    
    @abstractmethod
    def get_vars(self) -> Set[str]:
        """Return set of variables in this term."""
        pass

@dataclass
class Constant(Term):
    """Constant symbol."""
    name: str
    
    def __str__(self) -> str:
        return self.name
    
    def get_vars(self) -> Set[str]:
        return set()
    
    def __hash__(self):
        return hash(('const', self.name))
    
    def __eq__(self, other):
        return isinstance(other, Constant) and self.name == other.name

@dataclass
class Variable(Term):
    """Variable symbol."""
    name: str
    
    def __str__(self) -> str:
        return self.name
    
    def get_vars(self) -> Set[str]:
        return {self.name}
    
    def __hash__(self):
        return hash(('var', self.name))
    
    def __eq__(self, other):
        return isinstance(other, Variable) and self.name == other.name

@dataclass
class Function(Term):
    """Function application."""
    name: str
    args: List[Term] = field(default_factory=list)
    
    def __str__(self) -> str:
        if not self.args:
            return self.name
        return f"{self.name}({', '.join(str(arg) for arg in self.args)})"
    
    def get_vars(self) -> Set[str]:
        return set().union(*(arg.get_vars() for arg in self.args))
    
    def __hash__(self):
        return hash(('func', self.name, tuple(self.args)))
    
    def __eq__(self, other):
        return (isinstance(other, Function) and 
                self.name == other.name and 
                self.args == other.args)

@dataclass
class Predicate:
    """Predicate (atomic formula)."""
    name: str
    args: List[Term] = field(default_factory=list)
    
    def __str__(self) -> str:
        if not self.args:
            return self.name
        return f"{self.name}({', '.join(str(arg) for arg in self.args)})"
    
    def get_vars(self) -> Set[str]:
        return set().union(*(arg.get_vars() for arg in self.args))
    
    def __hash__(self):
        return hash(('pred', self.name, tuple(self.args)))
    
    def __eq__(self, other):
        return (isinstance(other, Predicate) and 
                self.name == other.name and 
                self.args == other.args)

print('✓ FOL term and predicate classes defined')

In [ ]:
# Example FOL expressions
print('=== FOL Syntax Examples ===')
print()

# Constants
socrates = Constant('Socrates')
plato = Constant('Plato')
print(f'Constant: {socrates}')

# Variables
x = Variable('x')
y = Variable('y')
print(f'Variable: {x}')

# Function
father = Function('fatherOf', [socrates])
print(f'Function: {father}')

# Nested function
grandfather = Function('fatherOf', [father])
print(f'Nested function: {grandfather}')

# Predicates
human_s = Predicate('Human', [socrates])
print(f'Predicate: {human_s}')

parent = Predicate('Parent', [socrates, plato])
print(f'Binary predicate: {parent}')

# With variables
human_x = Predicate('Human', [x])
print(f'Predicate with variable: {human_x}')

## 5.3 Unification

**Unification** is the key inference mechanism in first-order logic. It finds substitutions that make expressions identical.

### The Problem

Given two expressions, find a **substitution** (binding of variables to terms) that makes them identical.

**Examples**:

1. Unify P(x) and P(Socrates)
   - Substitution: {x → Socrates}
   - Result: P(Socrates)

2. Unify P(x, John) and P(Mary, y)
   - Substitution: {x → Mary, y → John}
   - Result: P(Mary, John)

3. Unify P(x, x) and P(John, Mary)
   - **Fails!** (x cannot be both John and Mary)

### Most General Unifier (MGU)

If multiple unifiers exist, we want the **most general** one.

**Example**: P(x, y) and P(z, z)
- {x → John, y → John, z → John} - unifier
- {x → z, y → z} - **MGU** (more general)

### Unification Algorithm

**Input**: Two expressions
**Output**: MGU or failure

**Steps**:
1. If both are identical, return empty substitution
2. If one is a variable, bind it to the other (occurs check!)
3. If both are functions/predicates:
   - Must have same name and arity
   - Recursively unify arguments
4. Otherwise, fail

**Occurs Check**: Prevent circular bindings
- Cannot unify x with f(x)
- Would create infinite term: f(f(f(...)))

In [ ]:
# Substitution (theta)
Substitution = Dict[str, Term]

def apply_substitution(term: Term, subst: Substitution) -> Term:
    """Apply substitution to a term."""
    if isinstance(term, Constant):
        return term
    
    elif isinstance(term, Variable):
        if term.name in subst:
            # Apply substitution recursively
            return apply_substitution(subst[term.name], subst)
        return term
    
    elif isinstance(term, Function):
        new_args = [apply_substitution(arg, subst) for arg in term.args]
        return Function(term.name, new_args)
    
    return term

def apply_substitution_pred(pred: Predicate, subst: Substitution) -> Predicate:
    """Apply substitution to a predicate."""
    new_args = [apply_substitution(arg, subst) for arg in pred.args]
    return Predicate(pred.name, new_args)

def occurs_check(var: str, term: Term) -> bool:
    """Check if variable occurs in term (prevents infinite structures)."""
    if isinstance(term, Variable):
        return var == term.name
    elif isinstance(term, Function):
        return any(occurs_check(var, arg) for arg in term.args)
    return False

def unify_terms(term1: Term, term2: Term, subst: Optional[Substitution] = None) -> Optional[Substitution]:
    """
    Unify two terms.
    
    Returns:
        Most general unifier (MGU) or None if unification fails
    """
    if subst is None:
        subst = {}
    
    # Apply current substitution
    term1 = apply_substitution(term1, subst)
    term2 = apply_substitution(term2, subst)
    
    # Identical terms
    if term1 == term2:
        return subst
    
    # Variable cases
    if isinstance(term1, Variable):
        if occurs_check(term1.name, term2):
            return None  # Occurs check failed
        new_subst = subst.copy()
        new_subst[term1.name] = term2
        return new_subst
    
    if isinstance(term2, Variable):
        if occurs_check(term2.name, term1):
            return None
        new_subst = subst.copy()
        new_subst[term2.name] = term1
        return new_subst
    
    # Function cases
    if isinstance(term1, Function) and isinstance(term2, Function):
        if term1.name != term2.name or len(term1.args) != len(term2.args):
            return None
        
        # Unify arguments
        current_subst = subst
        for arg1, arg2 in zip(term1.args, term2.args):
            current_subst = unify_terms(arg1, arg2, current_subst)
            if current_subst is None:
                return None
        return current_subst
    
    # Cannot unify
    return None

def unify_predicates(pred1: Predicate, pred2: Predicate) -> Optional[Substitution]:
    """Unify two predicates."""
    if pred1.name != pred2.name or len(pred1.args) != len(pred2.args):
        return None
    
    subst = {}
    for arg1, arg2 in zip(pred1.args, pred2.args):
        subst = unify_terms(arg1, arg2, subst)
        if subst is None:
            return None
    return subst

print('✓ Unification algorithm implemented')

In [ ]:
# Test unification
print('=== Unification Examples ===')
print()

# Example 1: P(x) and P(Socrates)
p1 = Predicate('P', [Variable('x')])
p2 = Predicate('P', [Constant('Socrates')])
print(f'Unify {p1} and {p2}')
subst = unify_predicates(p1, p2)
print(f'Result: {subst}')
print()

# Example 2: Parent(x, John) and Parent(Mary, y)
p3 = Predicate('Parent', [Variable('x'), Constant('John')])
p4 = Predicate('Parent', [Constant('Mary'), Variable('y')])
print(f'Unify {p3} and {p4}')
subst2 = unify_predicates(p3, p4)
print(f'Result: {subst2}')
print()

# Example 3: P(x, x) and P(John, Mary) - should fail
p5 = Predicate('P', [Variable('x'), Variable('x')])
p6 = Predicate('P', [Constant('John'), Constant('Mary')])
print(f'Unify {p5} and {p6}')
subst3 = unify_predicates(p5, p6)
print(f'Result: {"Failed" if subst3 is None else subst3}')

## 5.4 Forward and Backward Chaining in FOL

We extend the chaining algorithms from propositional logic to handle variables and unification.

### Knowledge Base Structure

**Facts**: Ground predicates (no variables)
- Human(Socrates)
- Human(Plato)

**Rules**: Implications with variables
- ∀x: Human(x) → Mortal(x)
- ∀x, y: Parent(x, y) → Ancestor(x, y)

### Forward Chaining

**Process**:
1. Start with known facts
2. Find rules whose premises unify with facts
3. Apply substitution and add new facts
4. Repeat until query is proved or no new facts

**Example**:
```
Facts: Human(Socrates)
Rules: Human(x) → Mortal(x)
Query: Mortal(Socrates)

Step 1: Unify Human(x) with Human(Socrates)
        Substitution: {x → Socrates}
Step 2: Apply to conclusion: Mortal(Socrates)
Step 3: Add to facts
Step 4: Query proved!
```

### Backward Chaining

**Process**:
1. Start with query (goal)
2. Find rules that conclude the goal
3. Recursively prove premises with substitution
4. Return success if all premises proved

Similar to Prolog!

In [ ]:
@dataclass
class Rule:
    """FOL rule: premises → conclusion"""
    premises: List[Predicate]
    conclusion: Predicate
    
    def __str__(self) -> str:
        if not self.premises:
            return str(self.conclusion)
        prem_str = ' ∧ '.join(str(p) for p in self.premises)
        return f'{prem_str} → {self.conclusion}'
    
    def get_vars(self) -> Set[str]:
        vars_set = self.conclusion.get_vars()
        for prem in self.premises:
            vars_set |= prem.get_vars()
        return vars_set

class FOLKnowledgeBase:
    """First-Order Logic Knowledge Base."""
    
    def __init__(self):
        self.facts: Set[Predicate] = set()
        self.rules: List[Rule] = []
    
    def tell_fact(self, predicate: Predicate):
        """Add a fact (ground predicate)."""
        self.facts.add(predicate)
    
    def tell_rule(self, rule: Rule):
        """Add a rule."""
        self.rules.append(rule)
    
    def forward_chain(self, query: Predicate, max_iterations: int = 100, 
                       verbose: bool = True) -> bool:
        """Forward chaining with unification."""
        if verbose:
            print('Forward Chaining:')
            print(f'Query: {query}')
            print()
        
        inferred = set(self.facts)
        
        for iteration in range(max_iterations):
            # Check if query already inferred
            for fact in inferred:
                if unify_predicates(query, fact) is not None:
                    if verbose:
                        print(f'✓ Query proved! Found {fact}')
                    return True
            
            new_facts = set()
            
            # Try to apply each rule
            for rule in self.rules:
                # Try all combinations of facts for premises
                self._apply_rule(rule, inferred, new_facts, verbose)
            
            if not new_facts:
                break
            
            inferred.update(new_facts)
        
        if verbose:
            print(f'✗ Cannot prove query')
        return False
    
    def _apply_rule(self, rule: Rule, facts: Set[Predicate], 
                     new_facts: Set[Predicate], verbose: bool):
        """Try to apply rule with current facts."""
        if not rule.premises:
            # Rule with no premises - conclusion is always true
            new_facts.add(rule.conclusion)
            return
        
        # Simple case: single premise
        if len(rule.premises) == 1:
            for fact in facts:
                subst = unify_predicates(rule.premises[0], fact)
                if subst is not None:
                    # Apply substitution to conclusion
                    new_conclusion = apply_substitution_pred(rule.conclusion, subst)
                    if new_conclusion not in facts:
                        new_facts.add(new_conclusion)
                        if verbose:
                            print(f'Applied {rule} with {subst}')
                            print(f'  Inferred: {new_conclusion}')
    
    def backward_chain(self, query: Predicate, subst: Optional[Substitution] = None,
                        depth: int = 0, verbose: bool = True) -> bool:
        """Backward chaining (goal-driven)."""
        if subst is None:
            subst = {}
            if verbose:
                print('Backward Chaining:')
                print(f'Goal: {query}')
                print()
        
        indent = '  ' * depth
        
        # Apply current substitution to query
        query = apply_substitution_pred(query, subst)
        
        # Check if query unifies with a fact
        for fact in self.facts:
            unified = unify_predicates(query, fact)
            if unified is not None:
                if verbose:
                    print(f'{indent}✓ {query} matches fact {fact}')
                return True
        
        # Try rules that conclude the query
        for rule in self.rules:
            unified = unify_predicates(query, rule.conclusion)
            if unified is not None:
                if verbose:
                    print(f'{indent}Trying rule: {rule}')
                
                # Try to prove all premises
                all_proved = True
                for premise in rule.premises:
                    premise_inst = apply_substitution_pred(premise, unified)
                    if not self.backward_chain(premise_inst, unified, depth + 1, verbose):
                        all_proved = False
                        break
                
                if all_proved:
                    if verbose and depth == 0:
                        print(f'\n✓ Query proved!')
                    return True
        
        if verbose and depth == 0:
            print(f'\n✗ Cannot prove query')
        return False

print('✓ FOL Knowledge Base implemented')

In [ ]:
# Example: Family relationships
print('=== FOL Knowledge Base Example ===')
print()

kb = FOLKnowledgeBase()

# Facts
kb.tell_fact(Predicate('Human', [Constant('Socrates')]))
kb.tell_fact(Predicate('Human', [Constant('Plato')]))
kb.tell_fact(Predicate('Parent', [Constant('John'), Constant('Mary')]))

# Rules
# Human(x) → Mortal(x)
kb.tell_rule(Rule(
    [Predicate('Human', [Variable('x')])],
    Predicate('Mortal', [Variable('x')])
))

# Parent(x, y) → Ancestor(x, y)
kb.tell_rule(Rule(
    [Predicate('Parent', [Variable('x'), Variable('y')])],
    Predicate('Ancestor', [Variable('x'), Variable('y')])
))

print('Knowledge Base:')
print('Facts:')
for fact in kb.facts:
    print(f'  {fact}')
print('Rules:')
for rule in kb.rules:
    print(f'  {rule}')
print()

# Query
print('\n--- Testing Forward Chaining ---')
query = Predicate('Mortal', [Constant('Socrates')])
kb.forward_chain(query)
print()

print('\n--- Testing Backward Chaining ---')
kb.backward_chain(query)

## Programming Tasks

### Task 1: Family Tree Reasoning (Medium)
Build a family tree knowledge base:
- Define parent, sibling, ancestor relations
- Implement transitive closure for ancestors
- Query for relationships

### Task 2: Prolog Interpreter (Hard)
Build a simple Prolog-style interpreter:
- Parse Prolog syntax
- Implement backward chaining with backtracking
- Support lists and arithmetic

### Task 3: Resolution Theorem Prover (Hard)
Implement resolution for FOL:
- Convert to CNF with Skolemization
- Implement resolution with unification
- Prove mathematical theorems

### Task 4: Natural Language to FOL (Medium-Hard)
Translate English to FOL:
- Parse simple sentences
- Extract entities and relations
- Generate FOL formulas

### Task 5: Expert System (Medium)
Design an expert system:
- Medical diagnosis or troubleshooting
- FOL rules for domain knowledge
- Interactive query interface
- Explain reasoning chains

## Summary

First-Order Logic extends propositional logic with:

### Key Additions

1. **Objects and Terms**: Constants, variables, functions
2. **Predicates**: Relations over objects
3. **Quantifiers**: ∀ (for all) and ∃ (exists)
4. **Unification**: Pattern matching with variable binding

### Comparison

| Feature | Propositional | First-Order |
|---------|---------------|-------------|
| Expressiveness | Limited | Rich |
| Objects | No | Yes |
| Variables | No | Yes |
| Quantifiers | No | Yes |
| Inference | Resolution | Resolution + Unification |
| Decidable | Yes | No (semi-decidable) |

### Inference Methods

**Forward Chaining**:
- Data-driven
- Derives all consequences
- Good for production systems

**Backward Chaining**:
- Goal-driven
- Focused on query
- Basis for Prolog

**Resolution**:
- Complete for FOL
- Requires CNF conversion
- Used in automated theorem provers

### Key Takeaways

1. **More Expressive**: Can represent complex knowledge
2. **Natural Representation**: Matches human reasoning
3. **Computational Cost**: Inference is harder than propositional
4. **Undecidable**: No algorithm guarantees termination
5. **Practical**: Used in real expert systems and databases

### Applications

- Expert systems (MYCIN, DENDRAL)
- Databases (SQL based on FOL)
- Logic programming (Prolog, Datalog)
- Semantic web (RDF, OWL)
- Natural language processing
- Automated theorem proving

**Next**: Machine learning and neural networks introduce statistical, data-driven approaches that complement logical reasoning.

## Further Reading

### Textbooks
- Aggarwal, C. C. (2021). *Artificial Intelligence: A Textbook*. Springer. [Chapter 5]
- Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.). [Chapters 8-9]
- Nilsson, N. J. (1998). *Artificial Intelligence: A New Synthesis*. Morgan Kaufmann.

### Logic Programming
- Sterling, L., & Shapiro, E. (1994). *The Art of Prolog* (2nd ed.). MIT Press.
- Clocksin, W. F., & Mellish, C. S. (2003). *Programming in Prolog* (5th ed.). Springer.

### Automated Reasoning
- Robinson, J. A. (1965). A Machine-Oriented Logic Based on the Resolution Principle. *JACM*.
- Chang, C. L., & Lee, R. C. T. (1973). *Symbolic Logic and Mechanical Theorem Proving*. Academic Press.

### Online Resources
- [SWI-Prolog](https://www.swi-prolog.org/) - Popular Prolog implementation
- [Automated Theorem Proving](http://www.cs.miami.edu/home/geoff/Courses/COMP6210-10M/) - Course materials
- [Stanford Logic Group](http://logic.stanford.edu/) - Research and resources